# Building a Expected Goals-Model

This Notebook shows the Machine Learning Pipeline of a Expected Goals-Model (xG).

## 1) Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import pyplot
from scipy import stats
import sklearn
from sklearn.feature_selection import SelectKBest, chi2, VarianceThreshold
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import confusion_matrix, accuracy_score, recall_score, precision_score, f1_score, brier_score_loss, log_loss
import seaborn as sns
import pickle

Train_Set = pd.read_excel('Trainingsset_Bio.xlsx')
Test_Set = pd.read_excel('Testset_Bio.xlsx')

The dataframes consist of the following variables:
1) support_leg_distance <br>
2) distance_to_goal <br>
3) angle_to_goal <br>
4) knee_angle_support <br>
5) arm_angle_support <br>
6) knee_angle_strike <br>
7) trunk_angle_strike <br>
8) ankle_angle_strike <br>
9) inclination_angle_strike <br>
and the target variable:
10) goal


All there is to do, is to change the name of the dataset xlsx

In [ ]:
print(Train_Set.shape)
Train_Set.describe()

In [ ]:
Test_Set.describe()

In [ ]:
plt.figure()
plt.hist(Train_Set.goal, bins=2)
plt.xlabel('Goals')
plt.ylabel('Total Quantity')

## 2) Data Preparation

### 2.1) Splitting the data

In [ ]:
dfs_train = np.split(Train_Set, [-1], axis=1)
dfs_test = np.split(Test_Set, [-1], axis=1)
X_train = dfs_train[0]
X_test = dfs_test[0]
Y_train = dfs_train[1]
Y_test = dfs_test[1]
print(X_train.shape, X_test.shape, Y_train.shape, Y_test.shape)

### 2.2) Oversampling

In [ ]:
train = pd.concat([X_train, Y_train], axis=1)
count_class_0, count_class_1 = train.goal.value_counts()
print(train['goal'].shape)
train_successful_goal_class_0 = train[train['goal'] == 0]
train_successful_goal_class_1 = train[train['goal'] == 1]
train_successful_goal_class_1_over = train_successful_goal_class_1.sample(count_class_0, replace=True)
train_successful_goal_over = pd.concat([train_successful_goal_class_0, train_successful_goal_class_1_over], axis=0)
X_train = train_successful_goal_over.iloc[:,:-1]
Y_train = train_successful_goal_over.iloc[:,-1]
print(X_train.shape)

## 3) Model Training

### 3.1) Logistic Regression

In [ ]:
LoRe = LogisticRegression(max_iter=10000, class_weight='balanced', random_state=12)
LoRe.fit(X_train, Y_train)
LoRe_pred_train = LoRe.predict(X_train)
LoRe_pred_test = LoRe.predict(X_test)
LoRe_prob_train = LoRe.predict_proba(X_train).transpose()[1].transpose()
LoRe_prob_test = LoRe.predict_proba(X_test).transpose()[1].transpose()
LoRe_importance = LoRe.coef_[0]
print(LoRe_prob_test)
print(sklearn.metrics.brier_score_loss(Y_test, LoRe_prob_test))

with open('LogReg_Bio.pkl','wb') as f:
    pickle.dump(LoRe,f)

sorted_idx = LoRe_importance.argsort()
y_ticks = np.arange(0, len(X_train.columns))
fig, ax = plt.subplots()
ax.set_title('Logistic Regression')
ax.barh(y_ticks, LoRe_importance[sorted_idx])
ax.set_yticks(y_ticks)
ax.set_yticklabels(X_train.columns[sorted_idx])
fig.tight_layout()
# plt.show()
plt.savefig('LogReg_feature importance.png')

### 3.2) XGBoost

In [ ]:
XGB = XGBClassifier(random_state=12)
XGB.fit(X_train, Y_train)
XGB_pred_train = XGB.predict(X_train)
XGB_pred_test = XGB.predict(X_test)
XGB_prob_train = XGB.predict_proba(X_train).transpose()[1].transpose()
XGB_prob_test = XGB.predict_proba(X_test).transpose()[1].transpose()
XGB_importance = XGB.feature_importances_
print(XGB_prob_test)
print(sklearn.metrics.brier_score_loss(Y_test, XGB_prob_test))

with open('XGB_Bio.pkl','wb') as f:
    pickle.dump(XGB,f)

sorted_idx = XGB_importance.argsort()
y_ticks = np.arange(0, len(X_train.columns))
fig, ax = plt.subplots()
ax.set_title('XGBoost')
ax.barh(y_ticks, XGB_importance[sorted_idx])
ax.set_yticks(y_ticks)
ax.set_yticklabels(X_train.columns[sorted_idx])
fig.tight_layout()
# plt.show()
plt.savefig('XGB_feature importance.png')

### 3.3) RandomForest

In [ ]:
RF = RandomForestClassifier(random_state=12)
RF.fit(X_train, Y_train)
RF_pred_train = RF.predict(X_train)
RF_pred_test = RF.predict(X_test)
RF_prob_train = RF.predict_proba(X_train).transpose()[1].transpose()
RF_prob_test = RF.predict_proba(X_test).transpose()[1].transpose()
RF_importance = RF.feature_importances_
print(RF_prob_test)
print(sklearn.metrics.brier_score_loss(Y_test, RF_prob_test))

with open('RF_Bio.pkl','wb') as f:
    pickle.dump(RF,f)

sorted_idx = RF_importance.argsort()
y_ticks = np.arange(0, len(X_train.columns))
fig, ax = plt.subplots()
ax.set_title('Random Forest')
ax.barh(y_ticks, RF_importance[sorted_idx])
ax.set_yticks(y_ticks)
ax.set_yticklabels(X_train.columns[sorted_idx])
fig.tight_layout()
# plt.show()
plt.savefig('RF_feature importance.png')

### 3.4) AdaBoostClassifier

In [ ]:
ADA = AdaBoostClassifier(random_state=12)
ADA.fit(X_train, Y_train)
ADA_pred_train = ADA.predict(X_train)
ADA_pred_test = ADA.predict(X_test)
ADA_prob_train = ADA.predict_proba(X_train).transpose()[1].transpose()
ADA_prob_test = ADA.predict_proba(X_test).transpose()[1].transpose()
ADA_importance = ADA.feature_importances_
print(ADA_prob_test)
print(sklearn.metrics.brier_score_loss(Y_test, ADA_prob_test))

with open('ADA_Bio.pkl','wb') as f:
    pickle.dump(ADA,f)

sorted_idx = ADA_importance.argsort()
y_ticks = np.arange(0, len(X_train.columns))
fig, ax = plt.subplots()
ax.set_title('ADABoost Classifier')
ax.barh(y_ticks, ADA_importance[sorted_idx])
ax.set_yticks(y_ticks)
ax.set_yticklabels(X_train.columns[sorted_idx])
fig.tight_layout()
# plt.show()
plt.savefig('ADA_feature importance.png')

## 4) Evaluation

### 4.1) Logistic Regression

In [ ]:
confusion_matrix_test = sklearn.metrics.confusion_matrix(Y_test, LoRe_pred_test)
ax = sns.heatmap(confusion_matrix_test, annot=True, fmt="d", cmap='Blues')
ax.set_title('Confusion Matrix for xG-Model\n\n');
ax.set_xlabel('\nPredicted Values')
ax.set_ylabel('Actual Values ')
ax.xaxis.set_ticklabels(['False','True'])
ax.yaxis.set_ticklabels(['False','True'])
plt.figtext(1, 0.55, "F1-Score: " + str(sklearn.metrics.f1_score(Y_test , LoRe_pred_test)))
plt.figtext(1, 0.45, "Brier-Score-Loss: " + str(sklearn.metrics.brier_score_loss(Y_test , LoRe_prob_test)))
plt.show()

### 4.2) XGBoost

In [ ]:
confusion_matrix_test = sklearn.metrics.confusion_matrix(Y_test, XGB_pred_test)
ax = sns.heatmap(confusion_matrix_test, annot=True, fmt="d", cmap='Blues')
ax.set_title('Confusion Matrix for xG-Model\n\n');
ax.set_xlabel('\nPredicted Values')
ax.set_ylabel('Actual Values ')
ax.xaxis.set_ticklabels(['False','True'])
ax.yaxis.set_ticklabels(['False','True'])
plt.figtext(1, 0.55, "F1-Score: " + str(sklearn.metrics.f1_score(Y_test , XGB_pred_test)))
plt.figtext(1, 0.45, "Brier-Score-Loss: " + str(sklearn.metrics.brier_score_loss(Y_test, XGB_prob_test)))
plt.show()

### 4.3) RandomForest

In [ ]:
confusion_matrix_test = sklearn.metrics.confusion_matrix(Y_test, RF_pred_test)
ax = sns.heatmap(confusion_matrix_test, annot=True, fmt="d", cmap='Blues')
ax.set_title('Confusion Matrix for xG-Model\n\n');
ax.set_xlabel('\nPredicted Values')
ax.set_ylabel('Actual Values ')
ax.xaxis.set_ticklabels(['False','True'])
ax.yaxis.set_ticklabels(['False','True'])
plt.figtext(1, 0.55, "F1-Score: " + str(sklearn.metrics.f1_score(Y_test, RF_pred_test)))
plt.figtext(1, 0.45, "Brier-Score-Loss: " + str(sklearn.metrics.brier_score_loss(Y_test, RF_prob_test)))
plt.show()

### 4.4) AdaBoostClassifier

In [ ]:
confusion_matrix_test = sklearn.metrics.confusion_matrix(Y_test, ADA_pred_test)
ax = sns.heatmap(confusion_matrix_test, annot=True, fmt="d", cmap='Blues')
ax.set_title('Confusion Matrix for xG-Model\n\n');
ax.set_xlabel('\nPredicted Values')
ax.set_ylabel('Actual Values ')
ax.xaxis.set_ticklabels(['False','True'])
ax.yaxis.set_ticklabels(['False','True'])
plt.figtext(1, 0.55, "F1-Score: " + str(sklearn.metrics.f1_score(Y_test, ADA_pred_test)))
plt.figtext(1, 0.45, "Brier-Score-Loss: " + str(sklearn.metrics.brier_score_loss(Y_test, ADA_prob_test)))
plt.show()